# Session 2: the imbalance reveal

Exploration only. Everything the service depends on lives in `nw/triage`; this notebook imports it. Run from the repository root with the project environment.

In [ ]:
import json, collections
from pathlib import Path
import numpy as np

from nw.triage.train import (
    load_tickets,
    by_split,
    labels,
    build_pipeline,
    evaluate,
    choose_p0_threshold,
)
from nw.triage.model import TriageModel
from nw.triage.features import PRIORITIES

rows = load_tickets(Path("../data/tickets.jsonl"))
splits = by_split(rows)
len(rows), {k: len(v) for k, v in splits.items()}

## What the label looks like

In [ ]:
counts = collections.Counter(r["priority"] for r in rows)
for p in PRIORITIES:
    print(f"{p}: {counts[p]:5d}  {100 * counts[p] / len(rows):5.1f}%")

## The naive model

No class weights, plain argmax. Watch the P0 row of the confusion matrix.

In [ ]:
naive = build_pipeline(class_weight=None).fit(splits["train"], labels(splits["train"]))
m = TriageModel(
    pipeline=naive,
    classes=list(naive.named_steps["clf"].classes_),
    p0_threshold=1.01,
    metadata={"version": "naive"},
)
rep = evaluate(m, splits["test"])
print(
    f"accuracy {rep['accuracy']:.3f}  macro-F1 {rep['macro_f1']:.3f}  P0 recall {rep['p0_recall']:.3f}"
)
for label, row in zip(PRIORITIES, rep["confusion"]):
    print(label, row)

## Class weights

In [ ]:
weighted = build_pipeline(class_weight="balanced").fit(splits["train"], labels(splits["train"]))
m = TriageModel(
    pipeline=weighted,
    classes=list(weighted.named_steps["clf"].classes_),
    p0_threshold=1.01,
    metadata={"version": "weighted"},
)
rep = evaluate(m, splits["test"])
print(
    f"accuracy {rep['accuracy']:.3f}  macro-F1 {rep['macro_f1']:.3f}  P0 recall {rep['p0_recall']:.3f}"
)
for label, row in zip(PRIORITIES, rep["confusion"]):
    print(label, row)

## The threshold trade-off

Recall and precision for P0 as the threshold moves, on the validation split. Pick the point Northwind can live with.

In [ ]:
val = splits["val"]
proba = m.predict_proba(val)[:, 0]
y = labels(val) == "P0"
print(" t    recall  precision  flagged")
for t in np.linspace(0.05, 0.95, 19):
    pred = proba >= t
    rec = (pred & y).sum() / max(1, y.sum())
    prec = (pred & y).sum() / max(1, pred.sum())
    print(f"{t:.2f}   {rec:.2f}     {prec:.2f}     {pred.sum():4d}")
print("chosen:", choose_p0_threshold(proba, labels(val)))

## Calibration

Are the probabilities honest? Bucket the top probability and compare with how often the top guess was right.

In [ ]:
proba_all = m.predict_proba(splits["test"])
top = proba_all.max(axis=1)
correct = np.asarray([PRIORITIES[int(np.argmax(p))] for p in proba_all]) == labels(splits["test"])
for lo in np.arange(0.2, 1.0, 0.1):
    mask = (top > lo) & (top <= lo + 0.1)
    if mask.any():
        print(
            f"confidence {lo:.1f} to {lo + 0.1:.1f}: predicted {top[mask].mean():.2f}, actual {correct[mask].mean():.2f}, n={mask.sum()}"
        )

## Questions to bring back to the room

- Which threshold would you defend to the on-call lead, and with what number?
- Where do the P0 misses go in the confusion matrix, and what do those tickets have in common?